# Predictor Comparison Notebook

This notebook provides an interactive environment for comparing Bitcoin fee predictors.

## Contents
1. Load and compare result files
2. Metrics comparison tables
3. Visualization of predictions
4. Performance by horizon analysis

In [ ]:
# Setup - run this cell first
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import benchmark modules
from benchmarks.visualization import (
    plot_metrics_comparison,
    plot_performance_by_horizon,
    create_comparison_summary,
)

# Configure matplotlib for notebook
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print('Setup complete!')

## 1. Load Result Files

Load the pre-computed evaluation results from the `results/` directory.

In [ ]:
# Find all result files
results_dir = project_root / 'results'
result_files = list(results_dir.glob('*.json'))

print(f'Found {len(result_files)} result files:')
for f in sorted(result_files):
    print(f'  - {f.name}')

In [ ]:
# Load all results
results = []
for fp in result_files:
    with open(fp) as f:
        data = json.load(f)
        data['_filename'] = fp.name
        results.append(data)

print(f'Loaded {len(results)} results')

## 2. Metrics Comparison Table

Compare all predictors side-by-side in a table format.

In [ ]:
# Create comparison DataFrame
comparison_data = []
for r in results:
    comparison_data.append({
        'Predictor': r['predictor_name'],
        'Horizon': r['horizon'],
        'MAE': r['metrics'].get('mae'),
        'RMSE': r['metrics'].get('rmse'),
        'Inclusion': r['metrics'].get('inclusion_accuracy'),
        'Direction': r['metrics'].get('directional_accuracy'),
        'Snapshots': r.get('n_snapshots'),
    })

df = pd.DataFrame(comparison_data)

# Sort by MAE
df = df.sort_values('MAE')

# Format for display
styled = df.style.format({
    'MAE': '{:.2f}',
    'RMSE': '{:.2f}',
    'Inclusion': '{:.1%}',
    'Direction': '{:.1%}',
}).background_gradient(subset=['MAE'], cmap='RdYlGn_r')

styled

## 3. Visualizations

### 3.1 Overall Comparison Summary

In [ ]:
# Create comprehensive comparison figure
fig = create_comparison_summary(results)
plt.show()

### 3.2 Metrics Comparison (Bar Charts)

In [ ]:
# Compare specific metrics
fig = plot_metrics_comparison(
    results,
    metrics=['mae', 'rmse', 'inclusion_accuracy'],
    title='Predictor Comparison'
)
plt.show()

### 3.3 Performance by Horizon

In [ ]:
# Compare 3h vs 1d performance
fig = plot_performance_by_horizon(
    results,
    metric='mae',
    title='MAE by Prediction Horizon'
)
plt.show()

In [ ]:
# Inclusion accuracy by horizon
fig = plot_performance_by_horizon(
    results,
    metric='inclusion_accuracy',
    title='Inclusion Accuracy by Horizon'
)
plt.show()

## 4. Detailed Analysis

### 4.1 Best Performers

In [ ]:
# Find best performers
best_mae = min(results, key=lambda x: x['metrics'].get('mae', float('inf')))
best_inc = max(results, key=lambda x: x['metrics'].get('inclusion_accuracy', 0))

print('Best Performers:')
print('=' * 50)
print(f"\nLowest MAE: {best_mae['predictor_name']} ({best_mae['horizon']})")
print(f"  MAE: {best_mae['metrics']['mae']:.2f} sat/vbyte")
print(f"  Inclusion: {best_mae['metrics'].get('inclusion_accuracy', 0):.1%}")

print(f"\nHighest Inclusion: {best_inc['predictor_name']} ({best_inc['horizon']})")
print(f"  Inclusion: {best_inc['metrics'].get('inclusion_accuracy', 0):.1%}")
print(f"  MAE: {best_inc['metrics']['mae']:.2f} sat/vbyte")

### 4.2 3h vs 1d Comparison

In [ ]:
# Compare same predictor across horizons
predictors = set(r['predictor_name'] for r in results)

print('Horizon Comparison (3h vs 1d):')
print('=' * 60)

for pred in sorted(predictors):
    pred_results = [r for r in results if r['predictor_name'] == pred]
    if len(pred_results) < 2:
        continue
    
    print(f"\n{pred}:")
    for r in sorted(pred_results, key=lambda x: x['horizon']):
        mae = r['metrics'].get('mae', float('nan'))
        inc = r['metrics'].get('inclusion_accuracy', 0)
        print(f"  {r['horizon']}: MAE={mae:.2f}, Inclusion={inc:.1%}")

## 5. Custom Comparison

Select specific predictors to compare.

In [ ]:
# Customize this list to compare specific predictors
# Example: compare only 3h predictions
selected_results = [r for r in results if r['horizon'] == '3h']

if selected_results:
    fig = plot_metrics_comparison(
        selected_results,
        title='3-Hour Horizon Comparison'
    )
    plt.show()
else:
    print('No 3h results found')

## 6. Save Comparison Report

Optionally save the comparison to a file.

In [ ]:
# Save comparison summary as CSV
output_path = results_dir / 'comparison_summary.csv'
df.to_csv(output_path, index=False)
print(f'Summary saved to: {output_path}')

---

## Notes

### Inclusion Criterion
The benchmark uses a **0.05 percentile** (5th percentile) inclusion criterion. A prediction is successful if it exceeds the 5th percentile of actual fee rates in the target block.

### Adding New Results
To add new predictor results:
1. Run evaluation: `python evaluate.py path/to/predictor.py --horizon 3h`
2. Results are saved to `results/` directory
3. Re-run this notebook to include new results